# Olist Dataset Profiling

Lightweight exploration for the raw Kaggle dataset `olistbr/brazilian-ecommerce`.

This notebook is exploratory only. Bronze remains raw, and reusable pipeline logic belongs under `src/`.

## Project Setup

The path setup below supports launching this notebook from the repository root, the project root, or the `notebooks/` directory.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd()
if (cwd / "src" / "ingestion").exists():
    PROJECT_ROOT = cwd
elif cwd.name == "notebooks" and (cwd.parent / "src" / "ingestion").exists():
    PROJECT_ROOT = cwd.parent
elif (cwd / "projects" / "03-retail-revenue-analytics" / "src" / "ingestion").exists():
    PROJECT_ROOT = cwd / "projects" / "03-retail-revenue-analytics"
else:
    raise RuntimeError("Launch this notebook from the project directory, notebook directory, or repository root.")

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ingestion.raw_inventory import discover_raw_files, get_raw_data_dir, list_supported_data_files
from processing.bronze.profiling import select_main_csv_file

raw_dir = get_raw_data_dir()
raw_dir

## Raw Directory

The directory below is the raw Bronze landing area. It should contain files downloaded by the ingestion job.

In [ ]:
raw_dir.exists(), raw_dir

## Raw File Inventory

Bronze inventories all landed raw files. Hidden Kaggle operational artifacts can appear here and are tracked for transparency.

In [ ]:
inventory = discover_raw_files(raw_dir)
inventory

## Supported CSV Files

Bronze v1 profiles only supported `.csv` files and excludes hidden operational paths from profiling.

In [ ]:
csv_files = list_supported_data_files(raw_dir)
csv_files

## Selected Main CSV

The selected file is the largest supported CSV by size, with path-based tie breaking. This is a raw-stage profiling rule only, not a final business-grain decision.

In [ ]:
selected_csv = select_main_csv_file(csv_files)
selected_csv

## DataFrame Preview

Read the selected raw CSV for exploratory inspection only. No pipeline logic is defined in this notebook.

In [ ]:
import pandas as pd

df = pd.read_csv(selected_csv)
df.head()

In [ ]:
df.shape

In [ ]:
list(df.columns)

## Pandas Info

Pandas inference is useful for first-pass inspection, but business type decisions are deferred to future Silver contracts.

In [ ]:
from io import StringIO

buffer = StringIO()
df.info(buf=buffer)
print(buffer.getvalue())

In [ ]:
df.isna().sum().sort_values(ascending=False)

In [ ]:
int(df.duplicated().sum())

## Scope Reminder

This notebook supports source understanding. It does not define Silver tables, facts, dimensions, revenue KPIs, DBT marts, orchestration, APIs, or dashboards.